# Stage 3 — Embedding

Streams the selected photos through the frozen BioCLIP v1 tower on the T4 and stores the
embeddings as Parquet. **This is the stage that needs the GPU** — set the runtime to T4
before starting.

Only embeddings persist. The raw image corpus is never stored; photos are streamed,
embedded and discarded.

Splitting is by **observation**, never by photo. Photos of one individual straddling
train and validation is the single most likely source of a falsely optimistic accuracy
number, and it is invisible in the metrics — everything just looks good.

Preprocessing is verified against BioCLIP's `open_clip_config.json` rather than assumed
from CLIP defaults: 224x224 bicubic, mean `[0.48145466, 0.4578275, 0.40821073]`, std
`[0.26862954, 0.26130258, 0.27577711]`, 512-d output.

> **Not yet implemented.** `lifelist-embed` gets written when there is somewhere to run
> it. The development sandbox has no GPU and cannot reach S3, so writing it blind would
> mean shipping untested download, retry and batching code into the one stage that costs
> hours to re-run. The cells below are the intended shape, deliberately commented out.


In [ ]:
#@title Mount Drive and install the package
from google.colab import drive
drive.mount('/content/drive')

CACHE = '/content/drive/MyDrive/life-list/cache'  #@param {type:"string"}
REPO  = '/content/life-list'                       #@param {type:"string"}

import os
os.makedirs(CACHE, exist_ok=True)

if not os.path.exists(REPO):
    !git clone https://github.com/StefanWiswedel/Life-List.git {REPO}
else:
    !cd {REPO} && git pull --ff-only

!pip -q install -e {REPO}/training
print('cache:', CACHE)


In [ ]:
#@title Confirm the GPU
!nvidia-smi


## Embed

In [ ]:
# !lifelist-embed \
#     --manifest {CACHE}/photo_manifest.parquet \
#     --model hf-hub:imageomics/bioclip \
#     --batch-size 256 \
#     --split-fractions 0.8 0.1 0.1 \
#     --seed 0 \
#     --cache-dir {CACHE}


## Assert the split has not leaked

This also runs inside the pipeline, not only here. The failure it guards against is both
invisible and flattering, which is why it gets a runtime assertion rather than a code
review.


In [ ]:
# import pandas as pd
# from lifelist_train.splits import Photo, assert_no_observation_leakage
#
# emb = pd.read_parquet(f'{CACHE}/embeddings.parquet')
# splits = {
#     name: [Photo(r.photo_id, r.observation_id, r.taxon_id) for r in group.itertuples()]
#     for name, group in emb.groupby('split')
# }
# assert_no_observation_leakage(splits)
# print('no observation straddles a split')
